### Notebook 05 — Phase A, Step A5: Synthetic Chat Generation
### Generate 3-turn customer support conversations using Gemini 2.5 Pro
### Strategy: Tiered template pools with dynamic financial number injection

In [2]:
import pandas as pd
import numpy as np
import json
import os
import time
from dotenv import load_dotenv
from google import genai

# Load featured dataset
df = pd.read_csv("../data/processed/sbdr_featured_dataset.csv")
print(f"Dataset loaded: {df.shape}")
print(f"\nDistress distribution:")
print(df['distress_level'].value_counts().sort_index())

# Verify key columns exist for number injection
injection_cols = ['LIMIT_BAL', 'BILL_AMT1', 'PAY_AMT1', 'PAY_0', 
                  'lc_dti_mean', 'lc_annual_inc_mean', 'sp_avg_monthly_spend',
                  'spending_volatility', 'distress_level', 'customer_id']
missing = [c for c in injection_cols if c not in df.columns]
print(f"\nInjection columns missing: {missing if missing else 'None — all present'}")

# Configure Gemini (new SDK)
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
print(f"\nGemini client initialized")

Dataset loaded: (30000, 85)

Distress distribution:
distress_level
high         4500
low         10253
moderate    12614
severe       2633
Name: count, dtype: int64

Injection columns missing: None — all present

Gemini client initialized


## Step 1: Define Conversation Generation Prompts
Each tier gets a system prompt that guides Gemini to produce realistic 3-turn chats.
Conversations include {placeholders} for financial numbers that get injected per-customer later.

In [3]:
# Prompt templates for each distress tier
# Each generates 3-turn conversations: Customer → Agent → Customer

tier_prompts = {
    "low": """You are generating synthetic customer support chat logs for a bank.
The customer has LOW financial distress — they are stable, paying on time, just has a routine inquiry.

Generate {batch_size} unique 3-turn conversations. Each conversation has:
- Turn 1 (Customer): A routine question or minor request (balance inquiry, card replacement, transaction check)
- Turn 2 (Agent): Professional, helpful response
- Turn 3 (Customer): Satisfied, brief acknowledgment

Rules:
- Tone: Calm, polite, business-like
- Use these placeholders naturally in the customer's messages where relevant:
  {{LIMIT_BAL}}, {{BILL_AMT}}, {{PAY_AMT}}, {{MONTHLY_SPEND}}
- Keep each turn to 1-3 sentences
- Vary the topics: balance checks, card issues, payment confirmations, account settings
- Make each conversation distinct — different opening, different topic

Return ONLY a valid JSON array. Each element:
{{"turn_1": "...", "turn_2": "...", "turn_3": "..."}}
""",

    "moderate": """You are generating synthetic customer support chat logs for a bank.
The customer has MODERATE financial distress — they are starting to feel pressure, may have missed a payment, asking about options.

Generate {batch_size} unique 3-turn conversations. Each conversation has:
- Turn 1 (Customer): Mentions concern about upcoming payment, asks about flexibility, slight worry in tone
- Turn 2 (Agent): Reassuring, offers options like payment date change or temporary reduction
- Turn 3 (Customer): Relieved but still cautious, agrees to explore options

Rules:
- Tone: Slightly anxious but cooperative, not panicked
- Use these placeholders naturally: {{LIMIT_BAL}}, {{BILL_AMT}}, {{PAY_AMT}}, {{MONTHLY_SPEND}}, {{DTI}}
- Keep each turn to 1-3 sentences
- Topics: late payment concern, requesting extension, budget tightening, asking about fees
- Make each conversation distinct

Return ONLY a valid JSON array. Each element:
{{"turn_1": "...", "turn_2": "...", "turn_3": "..."}}
""",

    "high": """You are generating synthetic customer support chat logs for a bank.
The customer has HIGH financial distress — they are struggling, may have lost income or faced a medical emergency, clearly stressed.

Generate {batch_size} unique 3-turn conversations. Each conversation has:
- Turn 1 (Customer): Reaches out about inability to pay, mentions hardship (job loss, medical bills, family emergency), emotional language
- Turn 2 (Agent): Empathetic, offers hardship program or restructuring options
- Turn 3 (Customer): Grateful but overwhelmed, expresses uncertainty about future, may ask follow-up about the program

Rules:
- Tone: Anxious, vulnerable, sometimes apologetic or frustrated
- Use these placeholders naturally: {{LIMIT_BAL}}, {{BILL_AMT}}, {{PAY_AMT}}, {{MONTHLY_SPEND}}, {{DTI}}, {{ANNUAL_INC}}
- Keep each turn to 2-4 sentences (these customers tend to explain more)
- Topics: job loss, medical emergency, divorce, reduced hours, unexpected expenses
- Show real emotional weight — this is someone in crisis reaching out for help
- Make each conversation distinct

Return ONLY a valid JSON array. Each element:
{{"turn_1": "...", "turn_2": "...", "turn_3": "..."}}
""",

    "severe": """You are generating synthetic customer support chat logs for a bank.
The customer has SEVERE financial distress — they may be completely unable to pay, emotionally shut down, avoidant, or in deep crisis.

Generate {batch_size} unique 3-turn conversations. Each conversation has:
- Turn 1 (Customer): Very short/avoidant OR emotionally distressed. May be responding to a bank outreach, not initiating. Mentions total inability to pay, or expresses hopelessness.
- Turn 2 (Agent): Very gentle, non-threatening, offers hardship restructuring or payment pause
- Turn 3 (Customer): Either minimal response ("ok", "I don't know"), emotional breakdown, or goes silent (just "..." or "I'll try")

Rules:
- Tone: Defeated, withdrawn, sometimes angry/defensive, sometimes silent
- Use these placeholders sparingly (these customers often don't reference specific numbers):
  {{BILL_AMT}}, {{PAY_AMT}}
- Keep customer turns SHORT (1-2 sentences max, sometimes just a few words)
- Topics: total payment stop, ignoring calls, feeling hopeless, mentions of depression/anxiety, anger at the system
- Some customers should be nearly non-communicative
- Make each conversation distinct

Return ONLY a valid JSON array. Each element:
{{"turn_1": "...", "turn_2": "...", "turn_3": "..."}}
"""
}

print("Tier prompts defined:")
for tier, prompt in tier_prompts.items():
    print(f"  {tier}: {len(prompt)} chars")

Tier prompts defined:
  low: 938 chars
  moderate: 979 chars
  high: 1176 chars
  severe: 1268 chars


In [5]:
# Test Gemini Flash
test_response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents="Return a JSON array with 2 objects, each having keys 'turn_1', 'turn_2', 'turn_3' with short placeholder text. Return ONLY valid JSON, no markdown."
)

print("Status: SUCCESS")
print(f"\nRaw text:\n{test_response.text[:500]}")

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 19.350911526s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.go

## Step 2: Generate Conversation Pools via Gemini
200 unique conversations per tier × 4 tiers = 800 templates total.
Batch size of 20 per API call = 40 calls total. ~10-15 min with rate limiting.